In [1]:
import pandas as pd
import numpy as np

def scan_shift_errors(file_path, lag_col='lag_1'):
    print(f"⏳ Đang đọc dữ liệu từ: {file_path} ...")
    # Chỉ load các cột cần thiết để tiết kiệm RAM
    cols_to_load = ['store_nbr', 'item_nbr', 'date', 'unit_sales', lag_col]
    
    try:
        df = pd.read_parquet(file_path, columns=cols_to_load)
    except Exception as e:
        return f"Lỗi đọc file: {e}. Vui lòng kiểm tra lại đường dẫn hoặc tên cột."
    
    print("Đang sắp xếp dữ liệu (O(n log n))...")
    df = df.sort_values(by=['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)
    
    print("Đang quét lỗi Shift() mù quáng (O(n))...")
    # 1. Kiểm tra xem dòng hiện tại có đang nằm trong cùng 1 nhóm (Cửa hàng - Sản phẩm) với dòng trước không
    same_group = (df['store_nbr'] == df['store_nbr'].shift(1)) & (df['item_nbr'] == df['item_nbr'].shift(1))
    
    # 2. Tính khoảng cách thời gian giữa dòng hiện tại và dòng trước
    days_diff = (df['date'] - df['date'].shift(1)).dt.days
    
    # 3. Tìm các điểm bị khuyết ngày (Gap)
    is_gap = same_group & (days_diff > 1)
    
    # 4. TRUY VẾT LỖI: 
    # Nếu bị khuyết ngày, đúng ra lag_1 phải bằng 0 (hoặc rỗng).
    # Nhưng nếu thuật toán cũ dùng df.groupby().shift(1), nó sẽ bốc y nguyên unit_sales của dòng trước đó gán vào.
    blind_shift_sales = df['unit_sales'].shift(1)
    
    # Sai số epsilon 1e-5 để so sánh số thập phân (float) một cách an toàn
    is_error = is_gap & (np.abs(df[lag_col] - blind_shift_sales) < 1e-5)
    
    # ================= THỐNG KÊ KẾT QUẢ =================
    total_rows = len(df)
    total_gaps = is_gap.sum()
    total_errors = is_error.sum()
    
    error_percentage = (total_errors / total_rows) * 100
    gap_error_percentage = (total_errors / total_gaps) * 100 if total_gaps > 0 else 0
    
    print("\n" + "=" * 60)
    print(f"📊 BÁO CÁO QUÉT LỖI TRÊN CỘT: [{lag_col}]")
    print("=" * 60)
    print(f"1. Tổng số dòng dữ liệu               : {total_rows:,.0f} dòng")
    print(f"2. Số dòng bị ngắt quãng (chứa Gap)   : {total_gaps:,.0f} dòng")
    print(f"3. Số dòng dính lỗi Shift ngớ ngẩn    : {total_errors:,.0f} dòng")
    print("-" * 60)
    print(f"Tỷ lệ lỗi trên TOÀN BỘ dữ liệu     : {error_percentage:.2f}%")
    print(f"Tỷ lệ lỗi trên CÁC DÒNG NGẮT QUÃNG : {gap_error_percentage:.2f}%")
    print("=" * 60)
    
    if gap_error_percentage > 80:
        print("💥 KẾT LUẬN CỦA HỆ THỐNG:")
        print("Hầu hết các điểm ngắt quãng đều bị copy/paste sai dữ liệu.")
    elif total_errors > 0:
        print("⚠️ Có phát hiện lỗi nhưng không bao phủ toàn bộ.")
    else:
        print("✅ Không phát hiện lỗi shift mù trên cột này. Dữ liệu sạch.")
        
    # Trả về 5 dòng lỗi làm bằng chứng để bạn tự xem lại
    if total_errors > 0:
        print("\n🔎 5 dòng lỗi tiêu biểu:")
        evidence_df = df[is_error].copy()
        evidence_df['days_diff'] = days_diff[is_error]
        return evidence_df[['store_nbr', 'item_nbr', 'date', 'unit_sales', lag_col, 'days_diff']].head(5)

scan_shift_errors('/kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet', lag_col='lag_7') #thay lag_col để check cột khác

⏳ Đang đọc dữ liệu từ: /kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet ...
Đang sắp xếp dữ liệu (O(n log n))...
Đang quét lỗi Shift() mù quáng (O(n))...

📊 BÁO CÁO QUÉT LỖI TRÊN CỘT: [lag_7]
1. Tổng số dòng dữ liệu               : 35,688,322 dòng
2. Số dòng bị ngắt quãng (chứa Gap)   : 0 dòng
3. Số dòng dính lỗi Shift ngớ ngẩn    : 0 dòng
------------------------------------------------------------
Tỷ lệ lỗi trên TOÀN BỘ dữ liệu     : 0.00%
Tỷ lệ lỗi trên CÁC DÒNG NGẮT QUÃNG : 0.00%
✅ Không phát hiện lỗi shift mù trên cột này. Dữ liệu sạch.


In [2]:
import pandas as pd
import numpy as np
import gc

print("BƯỚC 1: TẠO MA TRẬN 16 TARGETS TƯƠNG LAI...")
train_df = pd.read_parquet('/kaggle/input/notebooks/luminhanh/ba-model-prep-data/train_clean.parquet')

train_df['item_nbr'] = train_df['item_nbr'].astype(int)
print("Đang sắp xếp dữ liệu ...")
train_df = train_df.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)

# Dùng observed=True để dập tắt cảnh báo và tối ưu tốc độ
grp_sales = train_df.groupby(['store_nbr', 'item_nbr'], observed=True)['unit_sales']
grp_promo = train_df.groupby(['store_nbr', 'item_nbr'], observed=True)['onpromotion']

print("Đang dịch chuyển thời gian (Time-shifting) cho 16 ngày...")
for i in range(1, 17):
    train_df[f'target_{i}'] = grp_sales.shift(-i).astype('float32')
    train_df[f'promo_future_{i}'] = grp_promo.shift(-i).fillna(0).astype('int8')

print("Đang tính toán Tâm lý Khuyến mãi (Promo Streak 3 ngày)...")
# Tổng số ngày KM trong 3 ngày tới (Đặc trưng mới bổ sung, chưa có trong tập 61 features)
train_df['promo_streak_3'] = (train_df['promo_future_1'] + train_df['promo_future_2'] + train_df['promo_future_3']).astype('int8')

print("✅ Đã tạo xong Ma trận Target! (Dữ liệu hoàn toàn sạch và không bị trùng lặp)")

BƯỚC 1: TẠO MA TRẬN 16 TARGETS TƯƠNG LAI...
Đang sắp xếp dữ liệu ...
Đang dịch chuyển thời gian (Time-shifting) cho 16 ngày...
Đang tính toán Tâm lý Khuyến mãi (Promo Streak 3 ngày)...
✅ Đã tạo xong Ma trận Target! (Dữ liệu hoàn toàn sạch và không bị trùng lặp)


In [3]:
import pandas as pd
import numpy as np
import gc

print("BƯỚC 2: TẠO TẬP TEST BASE (CHỐT TẠI 15/08)...")
# Lấy ngày cuối cùng của Train làm móng dự báo. 
X_test_base = train_df[train_df['date'] == '2017-08-15'].copy()

# Xóa Target tương lai bị NaN do chưa xảy ra
cols_to_drop = [f'target_{i}' for i in range(1, 17)] + ['unit_sales']
X_test_base.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Đang đồng bộ thông tin Khuyến mãi tương lai từ file Test...")
test_raw = pd.read_parquet('/kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_clean.parquet') 

# Ép kiểu dữ liệu trước khi Pivot để Merge không bị lỗi
test_raw['store_nbr'] = test_raw['store_nbr'].astype(str)
test_raw['item_nbr'] = test_raw['item_nbr'].astype(float).fillna(-1).astype(int)

print("Đang Pivot ma trận Khuyến mãi...")
# Dùng pivot_table để tự động gom dòng trùng lặp (nếu có) bằng hàm max
test_promo = test_raw.pivot_table(
    index=['store_nbr', 'item_nbr'],
    columns='date',
    values='onpromotion',
    aggfunc='max' 
)

# Đổi tên cột cho khớp với ma trận Target
test_promo.columns = [f'promo_future_{i}' for i in range(1, 17)]
test_promo = test_promo.reset_index()

# Lắp ghép vào X_test_base
X_test_base.drop(columns=[f'promo_future_{i}' for i in range(1, 17)], inplace=True, errors='ignore')
X_test_base = X_test_base.merge(test_promo, on=['store_nbr', 'item_nbr'], how='left')

for i in range(1, 17):
    X_test_base[f'promo_future_{i}'] = X_test_base[f'promo_future_{i}'].fillna(0).astype('int8')

print("Đồng bộ Tâm lý Khuyến mãi cho Test Base...")
# Đổi tên thành 'promo_sum_future_7' cho khớp với file Train
promo_7_test = sum(X_test_base[f'promo_future_{i}'] for i in range(1, 8))
X_test_base['promo_sum_future_7'] = promo_7_test.astype('int8')

# KHÓA ĐỊNH DẠNG LẦN CUỐI
print("Đang khóa định dạng Category...")
# ⚡ Bổ sung 'item_nbr' và 'class' cho khớp với CatBoost
safe_cat_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']
for col in safe_cat_features:
    if col in X_test_base.columns:
        X_test_base[col] = X_test_base[col].astype(str).astype('category')

print(f"✅ X_test_base sẵn sàng. Shape: {X_test_base.shape}")

# Dọn rác
del test_promo, test_raw; gc.collect()

BƯỚC 2: TẠO TẬP TEST BASE (CHỐT TẠI 15/08)...
Đang đồng bộ thông tin Khuyến mãi tương lai từ file Test...
Đang Pivot ma trận Khuyến mãi...
Đồng bộ Tâm lý Khuyến mãi cho Test Base...
Đang khóa định dạng Category...
✅ X_test_base sẵn sàng. Shape: (167515, 77)


0

In [4]:
import optuna
from catboost import CatBoostRegressor, Pool
import numpy as np
import pandas as pd
import gc
import json
import os
import shutil
import warnings
warnings.filterwarnings('ignore')

print("🎯 KHỞI ĐỘNG OPTUNA: SĂN BỘ THAM SỐ VÀNG CHO 61 FEATURES...")

INPUT_PARAMS_FILE = '/kaggle/input/datasets/luminhanh/optuna/optuna_best_params_final.json'
WORKING_PARAMS_FILE = '/kaggle/working/optuna_best_params_final.json'

# Phục hồi JSON từ Input sang Working (Nếu có)
if not os.path.exists(WORKING_PARAMS_FILE) and os.path.exists(INPUT_PARAMS_FILE):
    shutil.copy(INPUT_PARAMS_FILE, WORKING_PARAMS_FILE)

# Đọc nội dung JSON
if os.path.exists(WORKING_PARAMS_FILE):
    with open(WORKING_PARAMS_FILE, 'r') as f:
        best_params_all = json.load(f)
else:
    best_params_all = {} 

representative_days = [2, 7, 14]

# Kiểm tra xem JSON đã có đủ kết quả cho CẢ 3 NGÀY chưa
all_days_done = all(f"Ngày_{i}" in best_params_all for i in representative_days)

if all_days_done:
    print("✅ TÌM THẤY BỘ THAM SỐ VÀNG CHO TẤT CẢ CÁC NGÀY TRONG JSON.")
else:
    # CHỈ CHẠY KHI JSON CÒN THIẾU NGÀY)
    print("File JSON chưa hoàn thiện. Tiến hành load và dọn dẹp dữ liệu...")
    if 'train_df' in globals():
        print("Đang chặn dữ liệu cũ (trước 05/2017)...")
        train_df = train_df[train_df['date'] >= '2017-05-01'].reset_index(drop=True)
        gc.collect()

        print("Đang dọn dẹp Inf và lấp NaN (Chỉ cho cột số)...")
        numeric_cols = train_df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            col_type = train_df[col].dtype
            train_df[col] = train_df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(col_type)

        print("Đang cắt ranh giới Validation (15/07)...")
        train_mask = train_df['date'] <= '2017-07-15'
        valid_mask = (train_df['date'] > '2017-07-15') & (train_df['date'] <= '2017-07-31')
        
        df_train = train_df[train_mask].copy()
        df_valid = train_df[valid_mask].copy()
        
        cols_to_drop = ['date', 'unit_sales', 'id']
        df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns], inplace=True, errors='ignore')
        df_valid.drop(columns=[c for c in cols_to_drop if c in df_valid.columns], inplace=True, errors='ignore')
        
        del train_df; gc.collect()
        print("Đã dọn dẹp tập train_df gốc.")
    elif 'df_train' in globals() and 'df_valid' in globals():
        print("✅ df_train và df_valid đã có sẵn trong RAM, tiếp tục chạy...")
    else:
        raise ValueError("❌ LỖI CRITICAL: Không tìm thấy dữ liệu!")

    weight_train = np.where(df_train['perishable'] == 1, 1.25, 1.0).astype(np.float32)
    weight_valid = np.where(df_valid['perishable'] == 1, 1.25, 1.0).astype(np.float32)

    safe_cat_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']

    for col in safe_cat_features:
        if col in df_train.columns:
            df_train[col] = df_train[col].astype(str).astype('category')
            df_valid[col] = df_valid[col].astype(str).astype('category')

    base_features = [c for c in df_train.columns if c not in safe_cat_features and 'target' not in c and 'promo_future' not in c]

    # CHẠY OPTUNA
    for i in representative_days:
        key_name = f"Ngày_{i}"
        
        # CHỐT CHẶN CHO TỪNG NGÀY: Nếu ngày này đã có trong JSON thì lướt qua luôn
        if key_name in best_params_all:
            print(f"\n========================================================")
            print(f"⏭️ {key_name} ĐÃ CÓ KẾT QUẢ TRONG JSON. BỎ QUA.")
            continue
            
        print(f"\n========================================================")
        print(f"🚀 ĐANG SĂN THAM SỐ CHO NHÓM: {key_name.upper()}/16")
        print(f"========================================================")
        
        features_for_model = base_features + safe_cat_features + [f'promo_future_{i}']
        target_train = df_train[f'target_{i}'].astype('float32')
        target_valid = df_valid[f'target_{i}'].astype('float32')
        
        print("Đang cắt ngẫu nhiên 30% dữ liệu Train...")
        sample_idx = df_train.sample(frac=0.3, random_state=42).index
        weight_train_sample = np.where(df_train.loc[sample_idx, 'perishable'] == 1, 1.25, 1.0).astype(np.float32)
        
        train_pool = Pool(
            df_train.loc[sample_idx, features_for_model], 
            label=target_train.loc[sample_idx], 
            cat_features=safe_cat_features, 
            weight=weight_train_sample
        )
        
        valid_pool = Pool(df_valid[features_for_model], label=target_valid, cat_features=safe_cat_features, weight=weight_valid)
        
        if i <= 3: max_iters = 1500; patience = 50
        elif i <= 10: max_iters = 2500; patience = 75
        else: max_iters = 4000; patience = 150
            
        def objective(trial):
            param = {
                'iterations': max_iters,
                'task_type': 'GPU',
                'loss_function': 'RMSE',
                'eval_metric': 'RMSE',
                'has_time': True,
                'max_ctr_complexity': 1 if i >= 6 else 2,
                'random_seed': 42,
                'border_count': 128, 
                
                'depth': trial.suggest_int('depth', 5, 7),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 3.0, 12.0),
                'random_strength': trial.suggest_float('random_strength', 0.1, 1.0),
                'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0)
            }
            
            model = CatBoostRegressor(**param)
            model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=patience, verbose=0)
            
            return model.get_best_score()['validation']['RMSE']

        study = optuna.create_study(
            study_name=f"catboost_study_{key_name}", 
            direction='minimize'
        )
        
        print(f"🔥 Bắt đầu chạy 20 trials cho {key_name}...")
        study.optimize(objective, n_trials=20)

        # Cập nhật kết quả vào bộ nhớ và TẠO/GHI ĐÈ FILE JSON NGAY LẬP TỨC
        best_params_all[key_name] = study.best_params
        print(f"\n🏆 Xong {key_name}! RMSE tốt nhất: {study.best_value:.5f}")
        
        with open(WORKING_PARAMS_FILE, 'w') as f:
            json.dump(best_params_all, f, indent=4)
        
        del train_pool, valid_pool, target_train, target_valid; gc.collect()

print("\n TỔNG KẾT BỘ THAM SỐ VÀNG CHO DỮ LIỆU:")
for k, v in best_params_all.items():
    print(f"👉 {k}: {v}")

🎯 KHỞI ĐỘNG OPTUNA: SĂN BỘ THAM SỐ VÀNG CHO 61 FEATURES...
File JSON chưa hoàn thiện. Tiến hành load và dọn dẹp dữ liệu...
Đang chặn dữ liệu cũ (trước 05/2017)...
Đang dọn dẹp Inf và lấp NaN (Chỉ cho cột số)...
Đang cắt ranh giới Validation (15/07)...
Đã dọn dẹp tập train_df gốc.

🚀 ĐANG SĂN THAM SỐ CHO NHÓM: NGÀY_2/16
Đang cắt ngẫu nhiên 30% dữ liệu Train...


[I 2026-04-25 15:53:06,824] A new study created in memory with name: catboost_study_Ngày_2


🔥 Bắt đầu chạy 20 trials cho Ngày_2...


[I 2026-04-25 16:02:31,706] Trial 0 finished with value: 0.5782222046215602 and parameters: {'depth': 6, 'learning_rate': 0.06769011683348491, 'l2_leaf_reg': 8.69077058483142, 'random_strength': 0.2981341609567162, 'bagging_temperature': 0.020609721070548814}. Best is trial 0 with value: 0.5782222046215602.
[I 2026-04-25 16:10:04,222] Trial 1 finished with value: 0.5854108896734064 and parameters: {'depth': 5, 'learning_rate': 0.011636813451406611, 'l2_leaf_reg': 10.247169468150002, 'random_strength': 0.10161466748209969, 'bagging_temperature': 0.002362138400994196}. Best is trial 0 with value: 0.5782222046215602.
[I 2026-04-25 16:19:34,481] Trial 2 finished with value: 0.5845460014101872 and parameters: {'depth': 6, 'learning_rate': 0.010263827896189731, 'l2_leaf_reg': 4.030691292797753, 'random_strength': 0.19588009521080388, 'bagging_temperature': 0.7113003235472329}. Best is trial 0 with value: 0.5782222046215602.
[I 2026-04-25 16:31:00,035] Trial 3 finished with value: 0.582158227


🏆 Xong Ngày_2! RMSE tốt nhất: 0.57757

🚀 ĐANG SĂN THAM SỐ CHO NHÓM: NGÀY_7/16
Đang cắt ngẫu nhiên 30% dữ liệu Train...


...


🏆 Xong Ngày_7! RMSE tốt nhất: 0.59804

🚀 ĐANG SĂN THAM SỐ CHO NHÓM: NGÀY_14/16
Đang cắt ngẫu nhiên 30% dữ liệu Train...


[I 2026-04-25 19:40:57,823] A new study created in memory with name: catboost_study_Ngày_14


🔥 Bắt đầu chạy 20 trials cho Ngày_14...


[I 2026-04-25 19:42:09,254] Trial 0 finished with value: 0.6182784694978476 and parameters: {'depth': 5, 'learning_rate': 0.05001301775472483, 'l2_leaf_reg': 5.3196771307191835, 'random_strength': 0.6303477245839179, 'bagging_temperature': 0.7055344025964289}. Best is trial 0 with value: 0.6182784694978476.
[I 2026-04-25 19:44:17,055] Trial 1 finished with value: 0.6164451099802343 and parameters: {'depth': 7, 'learning_rate': 0.018670742453614782, 'l2_leaf_reg': 5.6260736090552355, 'random_strength': 0.2031967952450624, 'bagging_temperature': 0.25102980225898697}. Best is trial 1 with value: 0.6164451099802343.
[I 2026-04-25 19:46:22,114] Trial 2 finished with value: 0.6154319782851326 and parameters: {'depth': 7, 'learning_rate': 0.04831666092246817, 'l2_leaf_reg': 9.214437776646108, 'random_strength': 0.9530054991240378, 'bagging_temperature': 0.7882735927299526}. Best is trial 2 with value: 0.6154319782851326.
[I 2026-04-25 19:48:10,909] Trial 3 finished with value: 0.6184321423019


🏆 Xong Ngày_14! RMSE tốt nhất: 0.61543

 TỔNG KẾT BỘ THAM SỐ VÀNG CHO DỮ LIỆU:
👉 Ngày_2: {'depth': 7, 'learning_rate': 0.07596963318365409, 'l2_leaf_reg': 6.868949705331496, 'random_strength': 0.9989890554621943, 'bagging_temperature': 0.024564040030996843}
👉 Ngày_7: {'depth': 7, 'learning_rate': 0.05771943598492748, 'l2_leaf_reg': 6.113374906566993, 'random_strength': 0.3198074844286317, 'bagging_temperature': 0.14727465882483237}
👉 Ngày_14: {'depth': 7, 'learning_rate': 0.04831666092246817, 'l2_leaf_reg': 9.214437776646108, 'random_strength': 0.9530054991240378, 'bagging_temperature': 0.7882735927299526}


In [5]:
from catboost import CatBoostRegressor, Pool
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import gc
import json
import os

print("MODEL CATBOOST...")

# =====================================================================
# CHỐT CHẶN BẢO VỆ RAM & DỌN DẸP DỮ LIỆU
# =====================================================================
if 'train_df' in globals():
# Chỉ giữ lại data 3 tháng gần nhất để Train, tránh OOM
    print("Đang chặn dữ liệu cũ (trước 05/2017)...")
    train_df = train_df[train_df['date'] >= '2017-05-01'].reset_index(drop=True)
    gc.collect()

    print("Đang dọn dẹp Inf và lấp hố NaN (Chỉ cho cột số)...")
    numeric_cols = train_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        col_type = train_df[col].dtype
        train_df[col] = train_df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(col_type)

    print("Đang cắt ranh giới Validation (15/07)...")
    train_mask = train_df['date'] <= '2017-07-15'
    valid_mask = (train_df['date'] > '2017-07-15') & (train_df['date'] <= '2017-07-31')
    
    df_train = train_df[train_mask].copy()
    df_valid = train_df[valid_mask].copy()
    
    # Dọn sạch những cột quá khứ không cần thiết để nhường RAM
    cols_to_drop = ['date', 'unit_sales', 'id']
    df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns], inplace=True, errors='ignore')
    df_valid.drop(columns=[c for c in cols_to_drop if c in df_valid.columns], inplace=True, errors='ignore')
    
    del train_df; gc.collect()
    print("Đã dọn dẹp tập train_df gốc.")
elif 'df_train' in globals() and 'df_valid' in globals():
    print("✅ df_train và df_valid đã có sẵn trong RAM, tiếp tục chạy...")
else:
    raise ValueError("❌ LỖI CRITICAL: Không tìm thấy dữ liệu!")

# =====================================================================
# ĐỊNH NGHĨA LẠI TẬP FEATURE
# =====================================================================
# ÉP KIỂU XUỐNG float32 thay vì float64 mặc định
weight_train = np.where(df_train['perishable'] == 1, 1.25, 1.0).astype(np.float32)
weight_valid = np.where(df_valid['perishable'] == 1, 1.25, 1.0).astype(np.float32)

safe_cat_features = ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class']

base_features = [c for c in df_train.columns 
                 if c not in safe_cat_features 
                 and 'target' not in c 
                 and 'promo_future' not in c]

print(f"Tổng số lượng Feature sử dụng: {len(base_features) + len(safe_cat_features)} cột")

for col in safe_cat_features:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(str).astype('category')
        df_valid[col] = df_valid[col].astype(str).astype('category')
        if 'X_test_base' in globals() and col in X_test_base.columns:
            X_test_base[col] = X_test_base[col].astype(str).astype('category')

all_test_predictions = []

# =====================================================================
# ĐỌC BỘ THAM SỐ VÀNG TỪ OPTUNA (ƯU TIÊN WORKING TRƯỚC)
# =====================================================================
print("✅ Đang nạp bộ tham số vàng...")

INPUT_PARAMS_FILE = '/kaggle/input/datasets/luminhanh/optuna/optuna_best_params_final.json'
WORKING_PARAMS_FILE = '/kaggle/working/optuna_best_params_final.json'

if os.path.exists(WORKING_PARAMS_FILE):
    print(f"📥 TÌM THẤY BỘ THAM SỐ TẠI WORKING! Đang load file {WORKING_PARAMS_FILE}...")
    with open(WORKING_PARAMS_FILE, 'r') as f:
        loaded_optuna_params = json.load(f)
elif os.path.exists(INPUT_PARAMS_FILE):
    print(f"📥 TÌM THẤY BỘ THAM SỐ TẠI INPUT! Đang load file {INPUT_PARAMS_FILE}...")
    with open(INPUT_PARAMS_FILE, 'r') as f:
        loaded_optuna_params = json.load(f)
else:
    print("⚠️ KHÔNG TÌM THẤY FILE OPTUNA NÀO! Tự động chuyển về bộ tham số dự phòng...")
    loaded_optuna_params = {
      "Ngày_2": {
        "depth": 7,                
        "learning_rate": 0.079,
        "l2_leaf_reg": 4.5,
        "random_strength": 0.77,
        "bagging_temperature": 0.04
      },
      "Ngày_7": {
        "depth": 6,                
        "learning_rate": 0.06,
        "l2_leaf_reg": 9.3,
        "random_strength": 0.82,
        "bagging_temperature": 0.52
      },
      "Ngày_14": {
        "depth": 6,                
        "learning_rate": 0.057,
        "l2_leaf_reg": 5.3,
        "random_strength": 0.18,
        "bagging_temperature": 0.21
      }
    }

# Vòng lặp 16 ngày
for i in tqdm(range(1, 17), desc="🚀 Đang Train 16 Models CatBoost"):
    print(f"\n⏳ Đang huấn luyện & Dự báo Ngày {i}/16...")
    
    # Đã ĐỒNG BỘ hóa các mốc chia ngày (<=3, <=10) cho khớp với Optuna
    optuna_key = None
    if i <= 3: 
        optuna_key = "Ngày_2"      
        current_iters = 1500; current_patience = 50; current_ctr = 2
    elif i <= 10: 
        optuna_key = "Ngày_7"      
        current_iters = 2500; current_patience = 75; current_ctr = 1
    else: 
        optuna_key = "Ngày_14"     
        current_iters = 4000; current_patience = 150; current_ctr = 1

    cb_params = {
        'iterations': current_iters,           
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'has_time': True,
        'max_ctr_complexity': current_ctr, 
        'border_count': 254,         
        'random_seed': 42,
        'verbose': 500  
    }
    
    # Cập nhật thông số
    if optuna_key in loaded_optuna_params:
        cb_params.update(loaded_optuna_params[optuna_key]) 
    else:
        # Fallback lót đáy lần nữa nếu file JSON bị thiếu key bất kỳ
        if i <= 3: cb_params.update({'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 3.0})
        elif i <= 10: cb_params.update({'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0})
        else: cb_params.update({'depth': 6, 'learning_rate': 0.02, 'l2_leaf_reg': 8.0})

    features_for_model = base_features + safe_cat_features + [f'promo_future_{i}']
    
    target_train = df_train[f'target_{i}'].astype('float32')
    target_valid = df_valid[f'target_{i}'].astype('float32')
    
    train_pool = Pool(
        df_train[features_for_model], 
        label=target_train, 
        cat_features=safe_cat_features, 
        weight=weight_train
    )
    
    valid_pool = Pool(
        df_valid[features_for_model], 
        label=target_valid, 
        cat_features=safe_cat_features, 
        weight=weight_valid
    )
    
    model = CatBoostRegressor(**cb_params)
    model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=current_patience)
    
    best_score = model.get_best_score()['validation']['RMSE']
    print(f"   -> ✅ Valid RMSE (Ngày {i}): {best_score:.4f} | Vòng lặp chốt: {model.get_best_iteration()} | Depth: {cb_params.get('depth')}")
    
    # GIẢI PHÓNG RAM NGAY LẬP TỨC TRƯỚC KHI PREDICT
    del train_pool, valid_pool, target_train, target_valid
    gc.collect()

    pred = model.predict(X_test_base[features_for_model])
    
    pred = np.clip(pred, 0, 10)  
    pred = np.expm1(pred)        
    
    pred_df = X_test_base[['store_nbr', 'item_nbr']].copy()
    pred_df['date'] = pd.to_datetime('2017-08-15') + pd.Timedelta(days=i)
    pred_df['unit_sales'] = pred
    all_test_predictions.append(pred_df)
    
    del model, pred_df; gc.collect()

print("\n🎉 HOÀN TẤT 16 MODELS! Đang xử lý file nộp bài...")

final_pred_df = pd.concat(all_test_predictions, ignore_index=True)

test_raw = pd.read_parquet('/kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_clean.parquet')

test_raw['date'] = pd.to_datetime(test_raw['date'])
final_pred_df['date'] = pd.to_datetime(final_pred_df['date'])
test_raw['store_nbr'] = test_raw['store_nbr'].astype(str)
final_pred_df['store_nbr'] = final_pred_df['store_nbr'].astype(str)
test_raw['item_nbr'] = test_raw['item_nbr'].astype(float).fillna(-1).astype(int)
final_pred_df['item_nbr'] = final_pred_df['item_nbr'].astype(int)

submission = test_raw[['id', 'store_nbr', 'item_nbr', 'date']].merge(
    final_pred_df, 
    on=['store_nbr', 'item_nbr', 'date'], 
    how='left'
)

if 'days_since_last_sale' in X_test_base.columns:
    print("Đang xoá các mặt hàng ngừng kinh doanh...")
    dead_items = X_test_base[X_test_base['days_since_last_sale'] >= 14][['store_nbr', 'item_nbr']].copy()
    dead_items['store_nbr'] = dead_items['store_nbr'].astype(str) 
    dead_mask = submission.set_index(['store_nbr', 'item_nbr']).index.isin(dead_items.set_index(['store_nbr', 'item_nbr']).index)
    submission.loc[dead_mask, 'unit_sales'] = 0

submission['unit_sales'] = submission['unit_sales'].fillna(0)
submission[['id', 'unit_sales']].to_csv('/kaggle/working/submission_catboost.csv', index=False)
print("🏆 Đã lưu file: /kaggle/working/submission_catboost.csv")

MODEL CATBOOST...
✅ df_train và df_valid đã có sẵn trong RAM, tiếp tục chạy...
Tổng số lượng Feature sử dụng: 60 cột
✅ Đang nạp bộ tham số vàng...
📥 TÌM THẤY BỘ THAM SỐ TẠI WORKING! Đang load file /kaggle/working/optuna_best_params_final.json...


🚀 Đang Train 16 Models CatBoost:   0%|          | 0/16 [00:00<?, ?it/s]


⏳ Đang huấn luyện & Dự báo Ngày 1/16...
0:	learn: 1.0286527	test: 1.0211702	best: 1.0211702 (0)	total: 658ms	remaining: 16m 25s
500:	learn: 0.5759268	test: 0.5741096	best: 0.5740762 (494)	total: 4m 47s	remaining: 9m 32s
bestTest = 0.5734516513
bestIteration = 622
Shrink model to first 623 iterations.
   -> ✅ Valid RMSE (Ngày 1): 0.5735 | Vòng lặp chốt: 622 | Depth: 7

⏳ Đang huấn luyện & Dự báo Ngày 2/16...
0:	learn: 1.0286645	test: 1.0239253	best: 1.0239253 (0)	total: 634ms	remaining: 15m 50s
500:	learn: 0.5805396	test: 0.5799105	best: 0.5799063 (499)	total: 4m 45s	remaining: 9m 29s
bestTest = 0.5790013956
bestIteration = 761
Shrink model to first 762 iterations.
   -> ✅ Valid RMSE (Ngày 2): 0.5790 | Vòng lặp chốt: 761 | Depth: 7

⏳ Đang huấn luyện & Dự báo Ngày 3/16...
0:	learn: 1.0280797	test: 1.0239761	best: 1.0239761 (0)	total: 633ms	remaining: 15m 48s
500:	learn: 0.5825203	test: 0.5855453	best: 0.5855391 (499)	total: 4m 48s	remaining: 9m 34s
bestTest = 0.5843990506
bestIteration